# Example-06: TbT filtering (SVD & Hankel & RPCA)

In [1]:
# Import

import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import data_load
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# In this example frequency uncertanty is compared for different TbT filtering

In [4]:
# Compute reference frequency for test TbT data

# Load TbT

length = 4096

w = Window(length, 'cosine_window', 2.0, dtype=dtype, device=device)
d = Data.from_file(54, w, '../virtual_tbt.npy', dtype=dtype, device=device)

# Compute reference frequency

d.window_remove_mean()
d.window_apply()
f = Frequency(d)
f('parabola')
d.reset()
ref = f.frequency.mean().cpu().item()
print(ref)

0.463116901262685


In [5]:
# Set parameters

length = 1024

# Set noise std for each BPM
# Note, each BPM has a different value of noise sigma

s = 1.0E-4*torch.rand(54, dtype=dtype, device=device)

# Test TbT data with random noise

w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_file(54, w, '../virtual_tbt.npy', dtype=dtype, device=device)
d.add_noise(s)
d.data.copy_(d.work)

# Set filter instance

l = Filter(d)

# Set frequency instance

f = Frequency(d)

In [6]:
# Estimate noise (optimal SVD truncation)

_, n = l.estimate_noise(limit=64, cpu=True)

# Maximum noise estimation error

print(torch.max(100*torch.abs(n - s)/s).cpu().item())

9.636715942325775


In [7]:
# Compute frequencies without filtering

# Note, since error is random, i.e. deviation of mean from the reference
# The result should be judged mostly by frequency spread (assuming bias can be neglected)
# Depending on the noise level, windowing might (and in general does) increase spread

f = Frequency(d)

# Without window

f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: raw data without window')

# With window
# Note, in general, window will increase frequency spread, but reduce bias

d.window_remove_mean()
d.window_apply()
f('parabola')
d.reset()
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: raw data with window')

error: 1.16102183e-09   spread: 5.49676784e-07   case: raw data without window
error: 6.30374602e-08   spread: 8.9200318e-07    case: raw data with window


In [8]:
# Perform SVD filtering (full TbT)
# Note, filter modifies work container

d.reset()
l.filter_svd(rank=8, cpu=True)

# Compute frequencies with SVD filtering

# Without window
# Note, frequency spread is generaly reduced by SVD filtering

f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: SVD filtering without window')

# With window
# Note, filtering allows application of window without harmful effect on frequency spread
# In some cases frequency spread is further reduced

d.window_remove_mean()
d.window_apply()
f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: SVD filtering with window')

error: 2.09490148e-08   spread: 3.27909693e-07   case: SVD filtering without window
error: 7.96122089e-08   spread: 5.58779644e-07   case: SVD filtering with window


In [9]:
# Perform Hankel filtering
# Note, when applied to raw TbT, Hankel filtering performs worse than SVD

d.reset()
l.filter_hankel(rank=8, cpu=True, random=True, buffer=8, count=8)

# Compute frequencies with Hankel filtering

# Without window
f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: Hankel filtering without window')

# With window
d.window_remove_mean()
d.window_apply()
f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: Hankel filtering with window')

error: 4.44044681e-08   spread: 7.32287939e-07   case: Hankel filtering without window
error: 4.9462552e-08    spread: 7.5850303e-07    case: Hankel filtering with window


In [10]:
# Perform SVD & Hankel filtering
# If Hankel filtering is applied after SVD, the frequency spread is further slightly reduced on average

d.reset()
l.filter_svd(rank=8, cpu=True)
l.filter_hankel(rank=8, cpu=True, random=True, buffer=8, count=8)

# Compute frequencies with Hankel filtering

# Without window

f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: SVD & Hankel filtering without window')

# With window

d.window_remove_mean()
d.window_apply()
f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: SVD & Hankel filtering with window')

error: 4.18300841e-08   spread: 4.63847399e-07   case: SVD & Hankel filtering without window
error: 5.19099748e-08   spread: 4.56815949e-07   case: SVD & Hankel filtering with window


In [11]:
# Perform RPCA filtering
# Note, RPCA is not perfect for normal noise, large number of iterations is required in this case

d.reset()
l.filter_rpca(limit=2**10, factor=1.0E-15, cpu=True)

# Without window

f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: RPCA filtering without window')

# With window

d.window_remove_mean()
d.window_apply()
f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: RPCA filtering with window')

error: 7.43332906e-08   spread: 3.34987187e-07   case: RPCA filtering without window
error: 2.26745318e-08   spread: 3.92151024e-07   case: RPCA filtering with window
